## 1: load

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

path = "../data/ml-latest-small/"
ratings = pd.read_csv(path + "ratings.csv")
movies = pd.read_csv(path + "movies.csv")

ratings['date'] = pd.to_datetime(ratings["timestamp"], unit="s")

## 2: filter sparse movies and users

In [15]:
MIN_MOVIE_RATINGS = 5
MIN_USER_RATINGS = 20

movie_counts = ratings.groupby("movieId").size()
keep_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index
ratings_f = ratings[ratings["movieId"].isin(keep_movies)]

user_counts = ratings_f.groupby("userId").size()
keep_users = user_counts[user_counts >= MIN_USER_RATINGS].index
ratings_f = ratings_f[ratings_f["userId"].isin(keep_users)].copy()

print(f"Before: {len(ratings)} ratings, {ratings['userId'].nunique()} users, {ratings['movieId'].nunique()} movies")
print(f"After:  {len(ratings_f)} ratings, {ratings_f['userId'].nunique()} users, {ratings_f['movieId'].nunique()} movies")
print(f"Share of ratings kept: {len(ratings_f) / len(ratings):.1%}")

Before: 100836 ratings, 610 users, 9724 movies
After:  90137 ratings, 602 users, 3650 movies
Share of ratings kept: 89.4%


`Filtering:` keeping movies with at least 5 ratings dropped the catalogue `from 9,724 to 3,650 movies (62% of titles) but cost only 10.5% of the ratings`. 

All 610 users survived. We threw away the part of the long tail that has too little data to learn from.

## 3: per-user chronological split (70% train, 15% validation, 15% test)

In [16]:
ratings_f = ratings_f.sort_values(["userId", "timestamp"])
ratings_f["rank"] = ratings_f.groupby("userId").cumcount()
ratings_f["n"] = ratings_f.groupby("userId")["rating"].transform("size")
ratings_f["pos"] = ratings_f["rank"] / ratings_f["n"]

train = ratings_f[ratings_f["pos"] < 0.70]
val = ratings_f[(ratings_f["pos"] >= 0.70) & (ratings_f["pos"] < 0.85)]
test = ratings_f[ratings_f["pos"] >= 0.85]

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name}: {len(df)} ratings, {df['userId'].nunique()} users, "
          f"{df['date'].min().date()} to {df['date'].max().date()}")

train: 63357 ratings, 602 users, 1996-03-29 to 2018-09-17
val: 13525 ratings, 602 users, 1996-03-29 to 2018-09-17
test: 13255 ratings, 602 users, 1996-03-29 to 2018-09-24


**Filtering:** after removing movies with fewer than 5 ratings, 8 users fell below the 20-rating minimum and were dropped (137 ratings), leaving 602 users.

**Split sizes:** 63,357 train, 13,525 validation and 13,255 test (90,137 ratings in total). Every split contains the same 602 users.


#### Why a per-user chronological split?
A recommender's real job is to predict what a user will like *next*, based on what they have already rated. 

Splitting each user's ratings by time (oldest 70% for training, next 15% for validation, newest 15% for testing) reproduces that situation, whereas a random split lets the model train on ratings that happened *after* the ones it is tested on. That is `data leakage`, and it makes scores look better than they would be in production.

Splitting per user, rather than using one global cutoff date, also guarantees that every user has training data, so no one is left with nothing to learn from.



## 4: sanity checks

In [17]:
unseen = ~test["movieId"].isin(train["movieId"])
print(f"Test ratings on movies absent from train: {unseen.mean():.1%}")

print(train["rating"].mean(), val["rating"].mean(), test["rating"].mean())

Test ratings on movies absent from train: 0.4%
3.5584939312151773 3.4859519408502773 3.4882308562806488


`Cold items:` only 0.4% of test ratings are on movies that never appear in training, so the cold-item problem is small after filtering.

`Mean rating:` 3.56 in train versus 3.49 in validation and test. The gap is small, but later ratings run slightly lower than earlier ones, so a model that predicts the training average would be a little optimistic.

## 5: saving the splits

In [18]:
os.makedirs("../data/processed", exist_ok=True)
cols = ["userId", "movieId", "rating", "timestamp"]
train[cols].to_csv("../data/processed/train.csv", index=False)
val[cols].to_csv("../data/processed/val.csv", index=False)
test[cols].to_csv("../data/processed/test.csv", index=False)